In [2]:
# ==========================================================
# END-TO-END INTEGRATED YOLO + GCN TRAINING & EVALUATION
# ==========================================================
import os
import torch
import torch.nn as nn
import numpy as np
from ultralytics import YOLO

print("\n" + "═"*60)
print("  INTEGRATED YOLO + GCN END-TO-END TRAINING")
print("═"*60)

# ==========================================================
# 1. SETTINGS & PATHS (Using your exact directories)
# ==========================================================
WORKING_DIR = "G:/FRP 2026/AlphaDent/working"
RESULT_DIR = os.path.join(WORKING_DIR, "train_run_large_6407")
YOLO_WEIGHTS = os.path.join(RESULT_DIR, "weights/best.pt")
DATA_YAML = "G:/FRP 2026/AlphaDent/yaml/yolo_seg_train.yaml"
OUTPUT_DIR = "G:/FRP 2026/AlphaDent/outputs"

COOCCURRENCE_MATRIX_PATH = os.path.join(OUTPUT_DIR, "matrices", "A_norm.npy")

# ==========================================================
# 2. LOAD ADJACENCY MATRIX & YOLO MODEL
# ==========================================================
print(f"Loading YOLO model from {YOLO_WEIGHTS}...")
model = YOLO(YOLO_WEIGHTS)

print("Loading Co-occurrence Adjacency Matrix...")
if not os.path.exists(COOCCURRENCE_MATRIX_PATH):
    raise FileNotFoundError(f"Matrix not found at {COOCCURRENCE_MATRIX_PATH}")

A_static = np.load(COOCCURRENCE_MATRIX_PATH)
A_tensor = torch.tensor(A_static, dtype=torch.float32)

# ==========================================================
# 3. INJECT GCN INTO YOLO HEAD (THE SAFE "PRO" WAY)
# ==========================================================
# We create a PyTorch Module to wrap YOLO's class-prediction layers
class GCNClassWrapper(nn.Module):
    def __init__(self, original_cv3, A_matrix, alpha=0.20):
        super().__init__()
        self.original_cv3 = original_cv3
        self.alpha = alpha # 20% GCN / 80% YOLO Residual Connection
        # Register A as a buffer so it automatically moves to the GPU with the model!
        self.register_buffer('A', A_matrix)

    def forward(self, x):
        # 1. Get raw class logits from YOLO's original convolutions
        cls_logits = self.original_cv3(x)  # Shape: (Batch, Classes, Height, Width)

        # 2. Reshape for Matrix Multiplication
        B, C, H, W = cls_logits.shape
        cls_reshaped = cls_logits.permute(0, 2, 3, 1).reshape(-1, C)

        # 3. Graph Convolution
        cls_gcn = torch.matmul(cls_reshaped, self.A)

        # 4. Residual Connection (Keeps 80% of original YOLO features, adds 20% context)
        cls_gcn = (self.alpha * cls_gcn) + ((1 - self.alpha) * cls_reshaped)

        # 5. Reshape back to YOLO's expected format
        cls_gcn = cls_gcn.reshape(B, H, W, C).permute(0, 3, 1, 2)

        return cls_gcn

# Extract the final Detection/Segmentation Head
head = model.model.model[-1]

# YOLO has multiple detection scales. `cv3` contains the convolutions for class predictions.
# We replace each class convolution with our new GCN-wrapped version.
for i in range(len(head.cv3)):
    # Using alpha=0.20 for the safe residual connection we discussed
    head.cv3[i] = GCNClassWrapper(head.cv3[i], A_tensor, alpha=0.20)

print("Successfully Integrated GCN Graph Convolution into YOLO architecture!")

# ==========================================================
# 4. TRAIN THE INTEGRATED MODEL
# ==========================================================
print("\nStarting End-to-End Training of YOLO + Integrated GCN...")
print("This forces the YOLO backbone to learn the Co-occurrence Matrix natively.")

# Train the patched model
model.train(
    data=DATA_YAML,
    epochs=30,           # 30 epochs is great for fine-tuning
    batch=8,
    lr0=5e-5,            # Low learning rate so we don't destroy the pre-trained weights
    warmup_epochs=3,
    patience=15,
    cos_lr=True,
    name="yolo_gcn_integrated",
    project=os.path.join(OUTPUT_DIR, "integrated_training")
)

print("\nIntegrated Training Complete!")
new_weights_path = os.path.join(OUTPUT_DIR, 'integrated_training', 'yolo_gcn_integrated', 'weights', 'best.pt')
print(f"Your NEW improved weights are saved inside: {new_weights_path}")

# ==========================================================
# 5. AUTOMATIC EVALUATION
# ==========================================================
print("\n" + "═"*60)
print("  RUNNING FINAL EVALUATION ON INTEGRATED MODEL")
print("═"*60)

# Because the GCN is now built into the PyTorch architecture, we can just use 
# YOLO's native val() function. It will automatically use the GCN!
metrics = model.val(data=DATA_YAML, split='val', verbose=False)

print("\n" + "═"*60)
print("  FINAL INTEGRATED YOLO + GCN SCORES")
print("═"*60)
print(f"  mAP@50 (Box):  {metrics.box.map50:.4f}")
print(f"  mAP@50-95:     {metrics.box.map:.4f}")
if hasattr(metrics, 'seg'):
    print(f"  mAP@50 (Mask): {metrics.seg.map50:.4f}")
print("═"*60)
print("All tasks completed successfully!")


════════════════════════════════════════════════════════════
  INTEGRATED YOLO + GCN END-TO-END TRAINING
════════════════════════════════════════════════════════════
Loading YOLO model from G:/FRP 2026/AlphaDent/working\train_run_large_6407\weights/best.pt...
Loading Co-occurrence Adjacency Matrix...
Successfully Integrated GCN Graph Convolution into YOLO architecture!

Starting End-to-End Training of YOLO + Integrated GCN...
This forces the YOLO backbone to learn the Co-occurrence Matrix natively.
New https://pypi.org/project/ultralytics/8.4.37 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.16  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=G:/FRP 

KeyboardInterrupt: 